# Notebook 02 · Modelado

El dataset tiene un desbalance de 19:1 (95% sin ictus / 5% con ictus). Accuracy no es una métrica válida. Usamos AUC-ROC y REcall como métricas principales para minimizar falsos negativos.

---
## Indice
1. Setup e importaciones
2. Configuracion de MLflow
3. Dataset con Feature Engineering opcional
4. Preprocesado y Pipeline
5. Funciones auxiliares
6. `evaluate_with_cv_and_threshold` — CV manual con busqueda de threshold
7. `train_final_model` y `evaluate_on_test`
8. `log_to_mlflow` y `run_experiment`
9. Definicion de modelos
10. Ejecucion de experimentos
11. Comparativa local← reescrito
9. Modelo final — log completo y análisis

---
## 01 · Setup e importaciones

Cargamos todas las dependencias necesarias. Hay dos puntos clave aquí:
* Usamos <code>imblearn.pipeline.Pipeline</code> en lugar de <code>sklearn.pipeline.Pipeline</code> — esto es fundamental porque la versión de imbalanced-learn sabe que SMOTE solo debe aplicarse durante el <code>fit()</code>, nunca durante el <code>predict()</code> ni en el test set.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import (
    classification_report, roc_auc_score,
    recall_score, f1_score, precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, average_precision_score, fbeta_score
)

from xgboost import XGBClassifier

# ⚠️ imblearn.pipeline — no sklearn.pipeline
# Garantiza que SMOTE solo actua en fit(), nunca en predict()
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

import mlflow
import mlflow.sklearn

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TEST_SIZE    = 0.2
AUTHOR       = 'Jonathan'

print('✓ Setup completado')

/home/under/miniconda3/envs/py310/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


✓ Setup completado


/home/under/miniconda3/envs/py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---

## 02 · Configuración de MLflow

Usamos MLflow para hacer tracking de experimentos. Cada vez que se entrena un modelo, MLflow guarda automáticamente los parámetros, métricas y el modelo serializado. Esto nos permite:
* Reproducir cualquier experimento anterior con exactamente los mismos parámetros
* Comparar todos los modelos del equipo en una sola tabla
* Versionar los modelos y descargar el mejor para producción

Para ver la UI ejecutar en terminal (Solo con WSL): <code>mlflow ui --backend-store-uri file:///mnt/c/Users/under/Documents/F5/3_Projects/Stroker_project/mlruns --port 5001</code> y abrir <code>http://localhost:5001</code>


In [2]:
# ── Configuración MLflow ──
MLFLOW_URI      = 'file:///mnt/c/Users/under/Documents/F5/3_Projects/Stroker_project/mlruns'
EXPERIMENT_NAME = 'stroke_project_Experiment_dataset_RL'

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

ASSETS_DIR = Path().resolve().parent / 'assets' / 'V1.2'
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ MLflow configurado")

✓ MLflow configurado


---

## 03 · Carga y versiones del dataset

<code>get_dataset()</code> centraliza toda la lógica de limpieza en un solo lugar. Si el equipo decide cambiar. Solo hay que modificar esta función y todos los experimentos se actualizan automáticamente.

In [3]:
df = pd.read_csv('../data/raw/stroke_dataset.csv')
print(f'✓ {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'  Stroke=1: {df["stroke"].sum()} ({df["stroke"].mean()*100:.1f}%)')


def get_dataset(version: str = 'full', add_features: bool = False) -> pd.DataFrame:
    """
    Carga y limpia el dataset segun decisiones del EDA.

    Versiones
    ---------
    'full'   -> todos los pacientes
    'adults' -> solo age >= 17

    Feature Engineering (opcional)
    ------------------------------
    add_features=True añade variables derivadas. Se loguea en MLflow
    para comparar si mejoran el modelo.

    Features añadidas:
    - age_group     : binning de edad (child/young/adult/senior)
    - high_risk     : flag si tiene hipertension O enfermedad cardiaca
    - age_glucose   : interaccion edad x glucosa (ambas correlacionadas con ictus)
    """
    assert version in ('full', 'adults'), f"version debe ser 'full' o 'adults'"

    df_copy = df.copy()

    # Limpieza EDA: work_type='children' -> 'not_applied'
    df_copy.loc[df_copy['work_type'] == 'children', 'work_type'] = 'not_applied'

    # Sin columna 'group' — age ya es feature numerica continua

    if version == 'adults':
        before = len(df_copy)
        df_copy = df_copy[df_copy['age'] >= 17].copy()
        print(f'  Filtro adults: {before - len(df_copy)} filas eliminadas')

    if add_features:
        # 1. Binning de edad: captura relacion no lineal entre edad y riesgo
        df_copy['age_group'] = pd.cut(
            df_copy['age'],
            bins=[0, 17, 40, 60, 100],
            labels=['child', 'young', 'adult', 'senior']
        ).astype(str)  # string para que el ColumnTransformer lo trate como categorica

        # 2. Flag riesgo medico combinado
        df_copy['high_risk'] = (
            (df_copy['hypertension'] == 1) | (df_copy['heart_disease'] == 1)
        ).astype(int)

        # 3. Interaccion edad x glucosa (ambas con alta correlacion con stroke segun EDA)
        df_copy['age_glucose'] = df_copy['age'] * df_copy['avg_glucose_level']

    print(f"✓ Dataset '{version}' | add_features={add_features}: {df_copy.shape[0]:,} filas")
    return df_copy

✓ 4,981 filas x 11 columnas
  Stroke=1: 248 (5.0%)


---

## 04 · Preprocesado y Pipeline

Realizamos transformaciones distintas a cada grupo de columnas en paralelo:

* Numéricas → StandardScaler: centra en media 0 y escala a desviación 1. Necesario para Logistic Regression (sensible a la escala). 
* Random Forest y XGBoost no lo necesitan.
* Categóricas → OneHotEncoder: convierte categorías en columnas binarias. <code>handle_unknown='ignore'</code> evita errores si en producción llega una categoría no vista en entrenamiento.


El Pipeline encadena pasos en orden. Cuando llamamos a <code>pipeline.fit(X_train, y_train)</code>, cada paso se ejecuta en secuencia:

> prep → [smote] → model

In [4]:
def build_preprocessor(X_train: pd.DataFrame, model_ty="linear") -> ColumnTransformer:
    """
    Construye ColumnTransformer sobre X_train unicamente.
    Aprende media/std y categorias solo del train, nunca del test.

    model_ty
    --------
    'linear' -> StandardScaler en numericas (LR es sensible a la escala)
    'tree'   -> passthrough en numericas (RF y XGBoost no necesitan escalado)
    """
    cat_cols = X_train.select_dtypes(include="object").columns.tolist()
    num_cols = X_train.select_dtypes(exclude="object").columns.tolist()

    if model_ty == "linear":
        num_transformer = StandardScaler()
    else:
        num_transformer = "passthrough"

    preprocessor = ColumnTransformer([
        ("num", num_transformer, num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])

    return preprocessor


def build_pipeline(model, preprocessor, use_smote: bool = False, sampler=None) -> Pipeline:
    """
    Pipeline: prep -> [sampler] -> model

    - use_smote=True → usa SMOTE por defecto
    - sampler → permite pasar otro sampler (ej: ADASYN)
    """

    steps = [('prep', preprocessor)]

    # Prioridad: sampler explícito
    if sampler is not None:
        steps.append(('sampler', sampler))  
    elif use_smote:
        steps.append(('sampler', SMOTE(random_state=RANDOM_STATE)))

    steps.append(('model', model))
    return Pipeline(steps)


print('✓ build_preprocessor y build_pipeline definidos')

✓ build_preprocessor y build_pipeline definidos


---
## 05. Funciones auxiliares

Tres funciones que nos ayudan en nuestro pipeline

In [ ]:
def get_model_ty(model) -> str:
    """Devuelve el tipo de modelo para seleccionar el preprocesador adecuado."""
    return 'tree' if isinstance(model, (RandomForestClassifier, XGBClassifier)) else 'linear'


def get_scale_pos_weight(y) -> float:
    """
    Calcula el peso para compensar el desbalance en XGBoost.
    scale_pos_weight = n_negativos / n_positivos
    Equivalente a class_weight='balanced' en sklearn.
    Solo se usa cuando no hay SMOTE (si hay SMOTE, el dataset ya esta balanceado).
    """
    neg, pos = (y == 0).sum(), (y == 1).sum()
    return neg / pos


def find_best_threshold(y_true, y_prob) -> float:
    """
    Busca el umbral de decision que maximiza F-beta (beta=2).

    Por que F-beta con beta=2:
    - beta=2 pondera el Recall dos veces mas que la Precision.
    - En contexto medico de cribado, un falso negativo (no detectar
      un paciente en riesgo) es mucho mas costoso que un falso positivo.
    - beta=2 refleja esa asimetria de costes.

    Parametros
    ----------
    y_true : array de etiquetas reales
    y_prob : array de probabilidades predichas (clase positiva)

    """
    best_t     = 0.5
    best_score = 0.0

    for t in np.linspace(0.01, 0.99, 200):
        y_pred = (y_prob >= t).astype(int)

        # Evitar division por cero si el umbral es extremo
        if y_pred.sum() == 0:
            continue

        score = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
        if score > best_score:
            best_score = score
            best_t     = t

    return best_t


print('✓ Funciones auxiliares definidas')

✓ Funciones auxiliares definidas


---

## 06 · Función evaluate_with_cv_and_threshold

Esta es la funcion clave del notebook. Reemplaza evaluate_with_cv porque cross_validate no da acceso a las probabilidades por fold — y sin ellas no se puede buscar el threshold en validacion. El CV manual con cv.split() lo permite.

1. Entrena en train_fold
2. Obtiene y_prob en val_fold
3. Busca threshold optimo en val_fold con fbeta(beta=2)
4. Calcula metricas con ese threshold

> Promedia threshold y metricas → aplica en test


In [ ]:
def evaluate_with_cv_and_threshold(model, X_train: pd.DataFrame, y_train: pd.Series, use_smote: bool = False) -> dict:
    """
    Sin data leakage:
    El threshold se busca en el fold de VALIDACION de cada iteracion.
    El test set nunca participa en este proceso.
    El threshold final es el promedio de los 5 folds.

    Parametros
    ----------
    model     : estimador sklearn
    X_train   : features de entrenamiento (sin test)
    y_train   : target de entrenamiento (sin test)
    use_smote : bool
    
    """
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    fold_auc, fold_recall, fold_f1, fold_thresholds = [], [], [], []

    for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):

        X_tr  = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]
        y_tr  = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        # Clonar modelo para que cada fold sea independiente
        model_fold = clone(model)

        # XGBoost sin SMOTE necesita scale_pos_weight calculado sobre ESTE fold
        if isinstance(model_fold, XGBClassifier) and not use_smote:
            model_fold.set_params(scale_pos_weight=get_scale_pos_weight(y_tr))

        # Preprocesador y pipeline sobre el fold de entrenamiento
        preprocessor = build_preprocessor(X_tr, model_ty=get_model_ty(model))
        pipeline_fold = build_pipeline(model_fold, preprocessor, use_smote=use_smote)
        pipeline_fold.fit(X_tr, y_tr)

        # Probabilidades en validacion — nunca en test
        y_prob_val = pipeline_fold.predict_proba(X_val)[:, 1]

        # Buscar threshold optimo en ESTE fold de validacion
        best_t_fold = find_best_threshold(y_val, y_prob_val)
        y_pred_val  = (y_prob_val >= best_t_fold).astype(int)

        # Metricas de este fold con su threshold optimo
        fold_auc.append(roc_auc_score(y_val, y_prob_val))
        fold_recall.append(recall_score(y_val, y_pred_val, zero_division=0))
        fold_f1.append(f1_score(y_val, y_pred_val, zero_division=0))
        fold_thresholds.append(best_t_fold)

    # Este es el valor que aplicaremos en el test set
    avg_threshold = np.mean(fold_thresholds)

    return {
        'cv_auc':         np.mean(fold_auc),
        'cv_auc_std':     np.std(fold_auc),
        'cv_recall':      np.mean(fold_recall),
        'cv_f1':          np.mean(fold_f1),
        'best_threshold': round(avg_threshold, 4),
        'threshold_std':  round(np.std(fold_thresholds), 4),
    }


print('✓ evaluate_with_cv_and_threshold definida')

✓ evaluate_with_cv_and_threshold definida


---
## 07 · train_final_model y evaluate_on_test

1. Entrena el modelo final sobre todos los datos de train.
2. Evaluar el modelo con test

In [ ]:
def train_final_model(model, X_train: pd.DataFrame, y_train: pd.Series, use_smote: bool = False):
    """
    Entrena el modelo final sobre todos los datos de train.

    Una vez que el CV ha validado los hiperparametros y encontrado
    el threshold optimo, entrenamos sobre el train completo para
    aprovechar todos los datos disponibles antes de evaluar en test.

    Retorna
    -------
    pipeline : Pipeline ajustado
    scale_pos_weight : float o None
    """
    model_final      = clone(model)
    scale_pos_weight = None

    if isinstance(model_final, XGBClassifier) and not use_smote:
        scale_pos_weight = get_scale_pos_weight(y_train)
        model_final.set_params(scale_pos_weight=scale_pos_weight)

    preprocessor = build_preprocessor(X_train, model_ty=get_model_ty(model))
    pipeline      = build_pipeline(model_final, preprocessor, use_smote=use_smote)
    pipeline.fit(X_train, y_train)

    return pipeline, scale_pos_weight

print('✓ train_final_model definido')

In [ ]:

def evaluate_on_test(pipeline, X_test: pd.DataFrame, y_test: pd.Series, threshold: float = 0.5) -> tuple:
    """
    Evalua el modelo en el test set con el threshold encontrado en CV.

    El threshold viene de evaluate_with_cv_and_threshold, nunca se
    busca sobre el test set (eso seria data leakage).

    Retorna
    -------
    metrics  : dict con AUC, recall, F1, precision, PR-AUC
    y_pred   : predicciones binarias con el threshold aplicado
    y_prob   : probabilidades crudas (para graficos ROC)
    """
    y_prob = (
        pipeline.predict_proba(X_test)[:, 1]
        if hasattr(pipeline[-1], 'predict_proba')
        else pipeline.decision_function(X_test)
    )
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        'auc':       roc_auc_score(y_test, y_prob),
        'pr_auc':    average_precision_score(y_test, y_prob),
        'recall':    recall_score(y_test, y_pred, zero_division=0),
        'f1':        f1_score(y_test, y_pred, zero_division=0),
        'precision': precision_score(y_test, y_pred, zero_division=0),
    }
    return metrics, y_pred, y_prob


print('✓ Evaluate_on_test definidas')

✓ train_final_model y evaluate_on_test definidas


---

## 08 · log_to_mlflow y run_experiment

In [ ]:
def log_confusion_matrix(y_test, y_pred, title: str = ''):
    """
    Genera y loguea la confusion matrix como artifact en MLflow.
    """
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, y_pred),
        display_labels=['No Stroke', 'Stroke']
    ).plot(ax=ax)
    ax.set_title(title)
    path = ASSETS_DIR / f'Confusion_Matrix_{title}.png'
    plt.savefig(path, dpi=100, bbox_inches='tight')
    plt.close()
    mlflow.log_artifact(path)

In [ ]:
def log_roc_curve(y_test, y_prob, auc: float, title: str = ''):
    """
    Genera y loguea la curva ROC como artifact en MLflow.
    """
    fig, ax = plt.subplots(figsize=(5, 4))
    RocCurveDisplay.from_predictions(
        y_test, y_prob, ax=ax,
        name=f'AUC = {auc:.3f}'
    )
    ax.set_title(title)
    path = ASSETS_DIR / f'Curve_ROC{title}.png'
    plt.savefig(path, dpi=100, bbox_inches='tight')
    plt.close()
    mlflow.log_artifact(path)


In [ ]:
def log_to_mlflow( model, model_name, dataset_version, use_smote, add_features, cv_metrics, test_metrics,
                   auc_gap, author, pipeline, y_test, y_pred, y_prob, scale_pos_weight, log_artifacts: bool = True):
    """
    Centraliza todo el log a MLflow.

    Separacion de responsabilidades:
    - log_metric  : valores numericos comparables entre runs (metricas)
    - log_param   : hiperparametros del modelo y configuracion
    - set_tag     : metadatos no numericos (autor, tipo de modelo)
    - log_artifact: archivos (imagenes, modelos)
    """
    run_name = f"{model_name}_{dataset_version}_smote={use_smote}_feat={add_features}"

    with mlflow.start_run(run_name=run_name):

        # ── Metricas CV (robustas, promediadas sobre 5 folds) ──
        mlflow.log_metric('cv_auc',          round(cv_metrics['cv_auc'], 4))
        mlflow.log_metric('cv_auc_std',      round(cv_metrics['cv_auc_std'], 4))
        mlflow.log_metric('cv_recall',       round(cv_metrics['cv_recall'], 4))
        mlflow.log_metric('cv_f1',           round(cv_metrics['cv_f1'], 4))

        # ── Metricas test (evaluacion final con threshold de CV) ──
        mlflow.log_metric('auc',             round(test_metrics['auc'], 4))
        mlflow.log_metric('pr_auc',          round(test_metrics['pr_auc'], 4))
        mlflow.log_metric('recall',          round(test_metrics['recall'], 4))
        mlflow.log_metric('f1',              round(test_metrics['f1'], 4))
        mlflow.log_metric('precision',       round(test_metrics['precision'], 4))
        mlflow.log_metric('auc_gap_cv_test', round(auc_gap, 4))

        # ── Threshold ──
        mlflow.log_metric('best_threshold',  cv_metrics['best_threshold'])
        mlflow.log_metric('threshold_std',   cv_metrics['threshold_std'])

        # ── Hiperparametros del modelo ──
        params = model.get_params().copy()
        params.pop('scale_pos_weight', None)  # se loguea por separado
        mlflow.log_params({k: str(v) for k, v in params.items()})

        if scale_pos_weight is not None:
            mlflow.log_param('scale_pos_weight', round(scale_pos_weight, 3))
        mlflow.log_param('use_smote',   use_smote)
        mlflow.log_param('add_features', add_features)

        # ── Tags: metadatos, no son hiperparametros ──
        mlflow.set_tag('author',          author)
        mlflow.set_tag('model_type',      model_name)
        mlflow.set_tag('dataset_version', dataset_version)

        # ── Artifacts ──
        if log_artifacts:
            log_confusion_matrix(y_test, y_pred, title=run_name)
            log_roc_curve(y_test, y_prob, auc=test_metrics['auc'], title=run_name)
            mlflow.sklearn.log_model(pipeline, artifact_path=model_name)


print('✓ log_to_mlflow definida')

✓ log_to_mlflow definida


In [ ]:
def run_experiment(model, model_name: str, dataset_version: str = 'full', use_smote: bool = False,
                        author: str = AUTHOR, add_features: bool = False, log_artifacts: bool = True) -> dict:
    """
    Orquesta el flujo completo de un experimento.

    Flujo
    -----
    1. Carga datos (version + feature engineering)
    2. Split estratificado train/test
    3. CV manual: metricas robustas + threshold optimo sin data leakage
    4. Entrenamiento final sobre train completo
    5. Evaluacion en test con el threshold encontrado en CV
    6. Log en MLflow
    7. Retorna dict para comparativa local

    """
    print(f"\n{'='*55}")
    print(f'  {model_name} | {dataset_version} | smote={use_smote} | feat={add_features}')
    print(f"{'='*55}")

    # 1. Datos
    df_exp = get_dataset(dataset_version, add_features=add_features)
    X = df_exp.drop('stroke', axis=1)
    y = df_exp['stroke']

    # Split estratificado: preserva proporcion de clases en ambos splits
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    print(f'  Train: {len(X_train):,} | Test: {len(X_test):,} | Positivos train: {y_train.sum()}')

    # 2. CV con busqueda de threshold en validacion
    cv_metrics = evaluate_with_cv_and_threshold(model, X_train, y_train, use_smote)
    print(f"  CV AUC    : {cv_metrics['cv_auc']:.4f} +/- {cv_metrics['cv_auc_std']:.4f}")
    print(f"  CV Recall : {cv_metrics['cv_recall']:.4f}")
    print(f"  Threshold : {cv_metrics['best_threshold']:.4f} +/- {cv_metrics['threshold_std']:.4f}")

    # 3. Entrenamiento final
    pipeline, scale_pos_weight = train_final_model(model, X_train, y_train, use_smote)

    # 4. Evaluacion en test con el threshold de CV
    best_threshold = cv_metrics['best_threshold']
    test_metrics, y_pred, y_prob = evaluate_on_test(pipeline, X_test, y_test, threshold=best_threshold)

    # Gap CV vs test: indicador de overfitting (debe ser < 0.05)
    auc_gap = abs(cv_metrics['cv_auc'] - test_metrics['auc'])

    print(f"  TEST AUC  : {test_metrics['auc']:.4f} | GAP: {auc_gap:.4f} {'✓' if auc_gap < 0.05 else '⚠️ revisar'}")
    print(f"  TEST Rec  : {test_metrics['recall']:.4f}")
    print(f"  TEST F1   : {test_metrics['f1']:.4f}")
    print(f"  TEST Prec : {test_metrics['precision']:.4f}")
    print(f"  PR-AUC    : {test_metrics['pr_auc']:.4f}")
    print(f'\n{classification_report(y_test, y_pred, target_names=["Sin ictus", "Ictus"])}')

    # 5. Log MLflow
    log_to_mlflow(
        model, model_name, dataset_version, use_smote, add_features,
        cv_metrics, test_metrics, auc_gap, author,
        pipeline, y_test, y_pred, y_prob, scale_pos_weight,
        log_artifacts=log_artifacts
    )
    print('  ✓ Run registrado en MLflow')

    return {
        'model':          model_name,
        'dataset':        dataset_version,
        'smote':          use_smote,
        'add_features':   add_features,
        'threshold':      best_threshold,
        'threshold_std':  cv_metrics['threshold_std'],
        'cv_auc':         round(cv_metrics['cv_auc'], 4),
        'cv_recall':      round(cv_metrics['cv_recall'], 4),
        'cv_f1':          round(cv_metrics['cv_f1'], 4),
        'auc':            round(test_metrics['auc'], 4),
        'pr_auc':         round(test_metrics['pr_auc'], 4),
        'recall':         round(test_metrics['recall'], 4),
        'f1':             round(test_metrics['f1'], 4),
        'precision':      round(test_metrics['precision'], 4),
        'auc_gap':        round(auc_gap, 4),
        '_pipeline':      pipeline,  # para analisis posterior, no va al df
        '_y_test':        y_test,
        '_y_prob':        y_prob,
    }


print('✓ run_experiment definida')

✓ run_experiment definida


---
## 09 · Definicion de modelos

Aquí definimos los modelos y lanzamos los experimentos. Cada llamada a <code>run_experiment</code> genera un run independiente en MLflow con sus propias métricas, parámetros y el modelo serializado.

Guardamos todos los resultados en una lista para generar la comparativa al final (Local). Los mejores modelos con configuracion razonada se usarán en Optuna.


In [13]:
# ── Logistic Regression ──────────────────────────────────────────
# L1 (lasso): hace feature selection automatica, util con features correlacionadas
# class_weight='balanced': compensa el desbalance 19:1 ajustando pesos internamente
lr_model = LogisticRegression(
    penalty='l1',
    solver='liblinear',  # unico solver compatible con L1
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_STATE
)

# ── Random Forest ────────────────────────────────────────────────
# Tres variantes para comparar: profundidad libre, controlada y superficial
rf_base = RandomForestClassifier(
    n_estimators=200, max_depth=None,
    class_weight='balanced', random_state=RANDOM_STATE
)
rf_controlled = RandomForestClassifier(
    n_estimators=300, max_depth=10, min_samples_split=10,
    class_weight='balanced', random_state=RANDOM_STATE
)
rf_shallow = RandomForestClassifier(
    n_estimators=200, max_depth=5,
    class_weight='balanced', random_state=RANDOM_STATE
)

# ── XGBoost ──────────────────────────────────────────────────────
# scale_pos_weight se calcula dinamicamente en run_experiment
# Tres variantes: base, deep y regularizado
xgb_base = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=RANDOM_STATE
)
xgb_deep = XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=RANDOM_STATE
)
xgb_reg = XGBClassifier(
    n_estimators=400, max_depth=4, learning_rate=0.05,
    reg_alpha=1.0, reg_lambda=2.0,  # regularizacion L1+L2 para reducir overfitting
    eval_metric='logloss', random_state=RANDOM_STATE
)

print('✓ Modelos definidos')
print('  LR: 1 variante | RF: 3 variantes | XGBoost: 3 variantes')

✓ Modelos definidos
  LR: 1 variante | RF: 3 variantes | XGBoost: 3 variantes


---
## 10 · Ejecucion de experimentos

Experimentos disenados para responder preguntas concretas:

* ¿Mejora el feature engineering? 
* ¿SMOTE ayuda o no? 
* ¿Que profundidad es mejor en RF?


In [20]:
results = []

# ── Pregunta 1: LR como baseline ────────────────────────────────
results.append(run_experiment(lr_model,      'LR',  'full', use_smote=False, add_features=False))
results.append(run_experiment(lr_model,      'LR',  'full', use_smote=False, add_features=True))
results.append(run_experiment(lr_model,      'LR',  'full', use_smote=True,  add_features=False))

# ── Pregunta 2: RF — profundidad vs generalizacion ───────────────
results.append(run_experiment(rf_base,       'RF',  'full', use_smote=False, add_features=False))
results.append(run_experiment(rf_controlled, 'RF',  'full', use_smote=False, add_features=False))
results.append(run_experiment(rf_shallow,    'RF',  'full', use_smote=False, add_features=False))

# ── Pregunta 3: RF con features — mejora? ────────────────────────
results.append(run_experiment(rf_base,       'RF',  'full', use_smote=False, add_features=True))
results.append(run_experiment(rf_controlled, 'RF',  'full', use_smote=False, add_features=True))

# ── Pregunta 4: XGBoost — variantes ─────────────────────────────
results.append(run_experiment(xgb_base,      'XGB', 'full', use_smote=False, add_features=False))
results.append(run_experiment(xgb_deep,      'XGB', 'full', use_smote=False, add_features=False))
results.append(run_experiment(xgb_reg,       'XGB', 'full', use_smote=False, add_features=False))

# ── Pregunta 5: XGBoost con features ────────────────────────────
results.append(run_experiment(xgb_base,      'XGB', 'full', use_smote=False, add_features=True))
results.append(run_experiment(xgb_reg,       'XGB', 'full', use_smote=False, add_features=True))

print(f'\n✓ {len(results)} experimentos completados')


  LR | full | smote=False | feat=False
✓ Dataset 'full' | add_features=False: 4,981 filas
  Train: 3,984 | Test: 997 | Positivos train: 198
  CV AUC    : 0.8379 +/- 0.0276
  CV Recall : 0.7986
  Threshold : 0.5478 +/- 0.0884
  TEST AUC  : 0.8412 | GAP: 0.0034 ✓
  TEST Rec  : 0.7400
  TEST F1   : 0.2442
  TEST Prec : 0.1462
  PR-AUC    : 0.1711

              precision    recall  f1-score   support

   Sin ictus       0.98      0.77      0.86       947
       Ictus       0.15      0.74      0.24        50

    accuracy                           0.77       997
   macro avg       0.56      0.76      0.55       997
weighted avg       0.94      0.77      0.83       997



2026/04/19 21:12:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/19 21:13:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  ✓ Run registrado en MLflow

  LR | full | smote=False | feat=True
✓ Dataset 'full' | add_features=True: 4,981 filas
  Train: 3,984 | Test: 997 | Positivos train: 198
  CV AUC    : 0.8391 +/- 0.0274
  CV Recall : 0.7223
  Threshold : 0.6059 +/- 0.0247
  TEST AUC  : 0.8467 | GAP: 0.0076 ✓
  TEST Rec  : 0.6600
  TEST F1   : 0.2588
  TEST Prec : 0.1610
  PR-AUC    : 0.1743

              precision    recall  f1-score   support

   Sin ictus       0.98      0.82      0.89       947
       Ictus       0.16      0.66      0.26        50

    accuracy                           0.81       997
   macro avg       0.57      0.74      0.58       997
weighted avg       0.94      0.81      0.86       997



2026/04/19 21:13:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/19 21:13:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  ✓ Run registrado en MLflow

  LR | full | smote=True | feat=False
✓ Dataset 'full' | add_features=False: 4,981 filas
  Train: 3,984 | Test: 997 | Positivos train: 198
  CV AUC    : 0.8355 +/- 0.0279
  CV Recall : 0.7985
  Threshold : 0.5350 +/- 0.0937
  TEST AUC  : 0.8373 | GAP: 0.0018 ✓
  TEST Rec  : 0.7400
  TEST F1   : 0.2434
  TEST Prec : 0.1457
  PR-AUC    : 0.1875

              precision    recall  f1-score   support

   Sin ictus       0.98      0.77      0.86       947
       Ictus       0.15      0.74      0.24        50

    accuracy                           0.77       997
   macro avg       0.56      0.76      0.55       997
weighted avg       0.94      0.77      0.83       997



...

  ✓ Run registrado en MLflow

  XGB | full | smote=False | feat=True
✓ Dataset 'full' | add_features=True: 4,981 filas
  Train: 3,984 | Test: 997 | Positivos train: 198
  CV AUC    : 0.7933 +/- 0.0321
  CV Recall : 0.7878
  Threshold : 0.1952 +/- 0.1328
  TEST AUC  : 0.8307 | GAP: 0.0374 ✓
  TEST Rec  : 0.8200
  TEST F1   : 0.2356
  TEST Prec : 0.1376
  PR-AUC    : 0.1827

              precision    recall  f1-score   support

   Sin ictus       0.99      0.73      0.84       947
       Ictus       0.14      0.82      0.24        50

    accuracy                           0.73       997
   macro avg       0.56      0.77      0.54       997
weighted avg       0.94      0.73      0.81       997



2026/04/19 21:17:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/19 21:17:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  ✓ Run registrado en MLflow

  XGB | full | smote=False | feat=True
✓ Dataset 'full' | add_features=True: 4,981 filas
  Train: 3,984 | Test: 997 | Positivos train: 198
  CV AUC    : 0.7937 +/- 0.0325
  CV Recall : 0.7365
  Threshold : 0.2168 +/- 0.1415
  TEST AUC  : 0.8149 | GAP: 0.0213 ✓
  TEST Rec  : 0.7200
  TEST F1   : 0.2338
  TEST Prec : 0.1395
  PR-AUC    : 0.1556

              precision    recall  f1-score   support

   Sin ictus       0.98      0.77      0.86       947
       Ictus       0.14      0.72      0.23        50

    accuracy                           0.76       997
   macro avg       0.56      0.74      0.55       997
weighted avg       0.94      0.76      0.83       997



2026/04/19 21:17:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/19 21:17:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  ✓ Run registrado en MLflow

✓ 13 experimentos completados


---

## 11  Comparativa

Comparamos los modelo para encontrar el que mejor se ajuste a lo planteado.

In [21]:
# Columnas para el df (excluimos las privadas que empiezan con _)
METRIC_COLS = ['model','dataset','smote','add_features','threshold',
               'cv_auc','cv_recall','cv_f1','auc','pr_auc','recall','f1','precision','auc_gap']

df_results = (
    pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')} for r in results])
    .sort_values('cv_auc', ascending=False)
    .reset_index(drop=True)
)

print('\n COMPARATIVA COMPLETA (ordenado por CV AUC)\n')
display(
    df_results[METRIC_COLS].style
    .background_gradient(subset=['cv_auc','cv_recall','cv_f1','auc','recall'], cmap='YlGn')
    .background_gradient(subset=['auc_gap'], cmap='YlOrRd')
    .format({
        col: '{:.4f}' for col in
        ['cv_auc','cv_recall','cv_f1','auc','pr_auc','recall','f1','precision','auc_gap','threshold']
    })
    .set_caption('auc_gap < 0.05 = sin overfitting | verde oscuro = mejor valor')
)


 COMPARATIVA COMPLETA (ordenado por CV AUC)



,model,dataset,smote,add_features,threshold,cv_auc,cv_recall,cv_f1,auc,pr_auc,recall,f1,precision,auc_gap
0,LR,full,False,True,0.6059,0.8391,0.7223,0.2765,0.8467,0.1743,0.6600,0.2588,0.1610,0.0076
1,LR,full,False,False,0.5478,0.8379,0.7986,0.2658,0.8412,0.1711,0.7400,0.2442,0.1462,0.0034
2,LR,full,True,False,0.5350,0.8355,0.7985,0.2615,0.8373,0.1875,0.7400,0.2434,0.1457,0.0018
3,RF,full,False,True,0.2474,0.8183,0.7932,0.2558,0.8343,0.1683,0.7200,0.2353,0.1406,0.0160
4,RF,full,False,False,0.5005,0.8149,0.7372,0.2398,0.8265,0.1517,0.8000,0.2446,0.1444,0.0117
5,RF,full,False,False,0.2927,0.8114,0.7278,0.2582,0.8323,0.1587,0.7400,0.2442,0.1462,0.0209
6,RF,full,False,True,0.0632,0.8015,0.7378,0.2420,0.8013,0.1546,0.7200,0.2432,0.1463,0.0002
7,RF,full,False,False,0.0612,0.7962,0.7324,0.2351,0.8094,0.1526,0.6600,0.2349,0.1429,0.0132
8,XGB,full,False,False,0.2257,0.7950,0.7627,0.2385,0.8263,0.1716,0.7400,0.2263,0.1336,0.0313
9,XGB,full,False,True,0.2168,0.7937,0.7365,0.2356,0.8149,0.1556,0.7200,0.2338,0.1395,0.0213


In [22]:
# ── Agrupaciones para responder las preguntas de diseño ──────────
print('\n─── FEATURE ENGINEERING: ¿mejora? ───')
print(df_results.groupby('add_features')[['cv_auc','cv_recall','cv_f1']].mean().round(4))

print('\n─── SMOTE: ¿ayuda? ───')
print(df_results.groupby('smote')[['cv_auc','cv_recall','cv_f1']].mean().round(4))

print('\n─── POR MODELO ───')
print(df_results.groupby('model')[['cv_auc','cv_recall','auc_gap']].mean().round(4))

print('\n─── INTERACCION MODELO x FEATURES ───')
print(df_results.groupby(['model','add_features'])[['cv_auc','cv_recall']].mean().round(4))


─── FEATURE ENGINEERING: ¿mejora? ───
              cv_auc  cv_recall   cv_f1
add_features                           
False         0.8089     0.7534  0.2476
True          0.8092     0.7555  0.2489

─── SMOTE: ¿ayuda? ───
       cv_auc  cv_recall   cv_f1
smote                           
False  0.8068     0.7505  0.2470
True   0.8355     0.7985  0.2615

─── POR MODELO ───
       cv_auc  cv_recall  auc_gap
model                            
LR     0.8375     0.7731   0.0043
RF     0.8085     0.7457   0.0124
XGB    0.7925     0.7514   0.0305

─── INTERACCION MODELO x FEATURES ───
                    cv_auc  cv_recall
model add_features                   
LR    False         0.8367     0.7986
      True          0.8391     0.7223
RF    False         0.8075     0.7325
      True          0.8099     0.7655
XGB   False         0.7918     0.7442
      True          0.7935     0.7622


In [23]:
# Comparativa de primeros modelos
df_results = pd.DataFrame(results)
df_results.sort_values("recall", ascending=False)

,model,dataset,smote,add_features,threshold,threshold_std,cv_auc,cv_recall,cv_f1,auc,pr_auc,recall,f1,precision,auc_gap,_pipeline,_y_test,_y_prob
11,XGB,full,False,True,0.1952,0.1328,0.7933,0.7878,0.2345,0.8307,0.1827,0.82,0.2356,0.1376,0.0374,"(ColumnTransformer(transformers=[('num', 'pass...",3385 0 1375 0 3277 0 4490 0 1374 ...,"[0.02373189, 0.023000985, 0.007137729, 0.00029..."
5,RF,full,False,False,0.5005,0.0459,0.8149,0.7372,0.2398,0.8265,0.1517,0.80,0.2446,0.1444,0.0117,"(ColumnTransformer(transformers=[('num', 'pass...",3385 0 1375 0 3277 0 4490 0 1374 ...,"[0.2320020158896906, 0.2945761948159537, 0.077..."
10,XGB,full,False,False,0.2355,0.1634,0.7913,0.7276,0.2470,0.8246,0.1757,0.78,0.2468,0.1466,0.0333,"(ColumnTransformer(transformers=[('num', 'pass...",3385 0 1375 0 3277 0 4490 0 1374 ...,"[0.06756189, 0.016384916, 0.0068698623, 0.0004..."
2,LR,full,True,False,0.5350,0.0937,0.8355,0.7985,0.2615,0.8373,0.1875,0.74,0.2434,0.1457,0.0018,"(ColumnTransformer(transformers=[('num', Stand...",3385 0 1375 0 3277 0 4490 0 1374 ...,"[0.12705384871989178, 0.2867312931272421, 0.04..."
0,LR,full,False,False,0.5478,0.0884,0.8379,0.7986,0.2658,0.8412,0.1711,0.74,0.2442,0.1462,0.0034,"(ColumnTransformer(transformers=[('num', Stand...",3385 0 1375 0 3277 0 4490 0 1374 ...,"[0.1535702122441379, 0.3632374780218973, 0.068..."
9,XGB,full,False,False,0.1213,0.1045,0.7891,0.7424,0.2350,0.8184,0.1539,0.74,0.2364,0.1407,0.0294,"(ColumnTransformer(transformers=[('num', 'pass...",3385 0 1375 0 3277 0 4490 0 1374 ...,"[0.017459508, 0.018311175, 0.0038929102, 0.000..."
8,XGB,full,False,False,0.2257,0.1279,0.7950,0.7627,0.2385,0.8263,0.1716,0.74,0.2263,0.1336,0.0313,"(ColumnTransformer(transformers=[('num', 'pass...",3385 0 1375 0 3277 0 4490 0 1374 ...,"[0.059124343, 0.06419224, 0.010164192, 0.00053..."
4,RF,full,False,False,0.2927,0.0487,0.8114,0.7278,0.2582,0.8323,0.1587,0.74,0.2442,0.1462,0.0209,"(ColumnTransformer(transformers=[('num', 'pass...",3385 0 1375 0 3277 0 4490 0 1374 ...,"[0.036443530912108106, 0.15507264274027205, 0...."
6,RF,full,False,True,0.0632,0.0183,0.8015,0.7378,0.2420,0.8013,0.1546,0.72,0.2432,0.1463,0.0002,"(ColumnTransformer(transformers=[('num', 'pass...",3385 0 1375 0 3277 0 4490 0 1374 ...,"[0.01, 0.045, 0.0, 0.0, 0.0, 0.0, 0.005, 0.0, ..."
7,RF,full,False,True,0.2474,0.0458,0.8183,0.7932,0.2558,0.8343,0.1683,0.72,0.2353,0.1406,0.0160,"(ColumnTransformer(transformers=[('num', 'pass...",3385 0 1375 0 3277 0 4490 0 1374 ...,"[0.027401829201177647, 0.09883031303264674, 0...."
